# Lesson 8.3 — Serverless & Containers: Lambda, ECS, EKS & Fargate

**Level: Advanced** | **Exam:** SAP-C02 Domain 2, Task 2.5; Domain 4, Task 4.4

## Compute Modernization Path

```
VM (EC2) → Containers (ECS/EKS) → Serverless (Lambda/Fargate)
          |                       |
          |<--- You manage OS, runtime
                                  |<--- AWS manages everything
```

## Lambda Key Facts
- **Max duration**: 15 minutes
- **Max memory**: 10,240 MB (CPU scales with memory)
- **Concurrency**: 1,000 simultaneous executions per region (default, increasable)
- **Cold start**: ~100ms-1s for first invocation (use Provisioned Concurrency to eliminate)
- **Deployment package**: 50 MB zip, 250 MB unzipped; use Lambda Layers for dependencies

## ECS vs EKS vs Fargate

| | ECS (Fargate) | ECS (EC2) | EKS (Fargate) | EKS (EC2) |
|--|-------------|----------|--------------|----------|
| **Managed infra** | Full | Partial | Full | Partial |
| **Cost** | $$ | $ | $$$ | $$ |
| **Kubernetes** | No | No | Yes | Yes |
| **Use when** | Simple containers, AWS-native | Need control of EC2 | K8s workloads, vendor-neutral | K8s + cost optimization |

## Fargate vs EC2 Launch Type
- **Fargate**: No EC2 to manage. Pay per vCPU/GB per second. Best for variable workloads.
- **EC2**: You manage instances. Better cost for stable high-CPU workloads. Need spot support.

## Event Sources for Lambda
- **Synchronous**: API Gateway, ALB, CloudFront (Lambda@Edge)
- **Asynchronous**: S3, SNS, EventBridge (Lambda retries on failure, DLQ available)
- **Polling**: SQS, DynamoDB Streams, Kinesis (Lambda polls, processes batches)

In [ ]:
import boto3, json
account_id = boto3.client('sts').get_caller_identity()['Account']
print(f'Account: {account_id}')

In [ ]:
# Inventory Lambda and container services
import boto3

lambda_client = boto3.client('lambda', region_name='us-east-1')
ecs = boto3.client('ecs', region_name='us-east-1')

# List Lambda functions
paginator = lambda_client.get_paginator('list_functions')
functions = []
for page in paginator.paginate():
    functions.extend(page['Functions'])

print(f"Lambda Functions ({len(functions)}):")
for fn in functions[:10]:
    print(f"  {fn['FunctionName']:40} {fn['Runtime']:15} Memory: {fn['MemorySize']}MB  Timeout: {fn['Timeout']}s")

# List ECS clusters
clusters = ecs.list_clusters()['clusterArns']
print(f"\nECS Clusters ({len(clusters)}):")
for cluster_arn in clusters:
    cluster = ecs.describe_clusters(clusters=[cluster_arn])['clusters'][0]
    running = cluster.get('runningTasksCount', 0)
    pending = cluster.get('pendingTasksCount', 0)
    print(f"  {cluster['clusterName']:30} Running: {running}  Pending: {pending}")

# Compute decision matrix
print("\n=== Compute Selection Guide ===")
decisions = [
    ("Event-driven, < 15 min, variable", "Lambda (Serverless)"),
    ("Long-running container, traffic spikes", "ECS + Fargate"),
    ("Kubernetes workloads, multi-cloud", "EKS (EC2 or Fargate)"),
    ("Stable, high-CPU containerized app", "ECS on EC2 (Spot for cost)"),
    ("ML inference, sub-second latency", "Lambda (Provisioned Concurrency) or ECS"),
    ("Stateful application with persistent storage", "EC2 or ECS with EFS"),
]
for use_case, recommendation in decisions:
    print(f"  {use_case:50} → {recommendation}")

## Exercise

In [ ]:
# TODO: Design a serverless architecture for a REST API that:
# - Handles 0-10,000 requests/second (highly variable)
# - Needs user authentication
# - Reads/writes to a database
# - Has some operations that take up to 5 minutes
# 
# Fill in each component with the appropriate AWS service:

architecture = {
    "api_layer": "TODO",                    # Route requests
    "auth": "TODO",                          # Authenticate users
    "short_functions": "TODO",               # Fast operations (< 15 min)
    "long_operations": "TODO",               # Operations > 15 min
    "database": "TODO",                      # Storage (hint: serverless DB)
    "async_processing": "TODO",              # Decouple long operations
    "notifications": "TODO"                  # Notify on completion
}

for component, service in architecture.items():
    print(f"{component:25}: {service}")

## Summary
- **Lambda**: Event-driven, sub-15-min operations, auto-scale to zero. Cold starts solvable with Provisioned Concurrency.
- **ECS + Fargate**: Containers without managing EC2. Pay per task. Best for variable containerized workloads.
- **EKS**: When you need Kubernetes portability or existing K8s expertise.
- **Lambda invocation modes**: Synchronous (ALB/APIGW), Asynchronous (S3/SNS), Polling (SQS/DynamoDB)
- **Step Functions**: Orchestrate long workflows combining Lambda + wait states

---
**Next:** [Lesson 8.4 — Application Integration: SQS, SNS, Step Functions & EventBridge](lesson-8.4-application-integration.ipynb)